# The Ransomware Decade: Jack Lichwa

**Source:** Ransomware Decade dataset (Zenodo record 15571866), `incidents.csv` and `schema.json`. Original record: https://zenodo.org/records/15571866

**What one row is:** one ransomware incident against one victim organization (`uuid` is unique per row).

**Focus:** four fields (attacker group, data exfiltration, operational impact, personal data affected), 2020 to 2024.

## 1. Load and sample

In [ ]:
import ast
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()
pd.set_option("display.max_columns", 30)

DATA_PATH = "../../data/ransomware_decade_incidents.csv"
raw_data = pd.read_csv(DATA_PATH)
print(raw_data.shape)
# Victim and join key, then the fields Ransomware Decade adds beyond Ransomware.live

raw_data.head()

In [ ]:
# Full-file overview: rows, columns, and how complete the key fields are
dates = pd.to_datetime(raw_data["incident-date"], errors="coerce")
overview = pd.DataFrame(
    [
        ("Rows (one row = one incident)", len(raw_data)),
        ("Columns", raw_data.shape[1]),
        ("Unique uuid values", raw_data["uuid"].nunique()),
        ("First incident date", dates.min().date()),
        ("Last incident date", dates.max().date()),
        ("Boolean columns", len(raw_data.select_dtypes("bool").columns)),
        ("Blank values in boolean columns", int(raw_data.select_dtypes("bool").isna().sum().sum())),
        ("Share with a Ransomware.live link", round(raw_data["ransomware.live-url"].notna().mean(), 3)),
    ],
    columns=["measure", "value"],
).set_index("measure")
overview

The Decade data has no attacker recorded for about half of the 2020 to 2024 incidents (48%). Payment outcome is left out because 79% of its values are unknown. The data can work only as supplementary data and cannot provide a complete view of ransomware incidents.

## 2. Cleaning

In [ ]:
# Keep only the fields the analysis uses
df = raw_data[["incident-date", "attacker", "data-exfiltration",
               "impact.category.Operational Impact", "affected-data.category.PII"]].copy()

df["incident_year"] = pd.to_datetime(df["incident-date"], errors="coerce").dt.year

# Attackers: parse the list text, label empty lists "Unattributed" (kept, not dropped)
df["attacker_list"] = df["attacker"].fillna("[]").apply(lambda s: ast.literal_eval(s) if s.strip() else [])
df["attacker_primary"] = df["attacker_list"].apply(lambda xs: xs[0] if xs else "Unattributed")

# True/False fields as readable text
df["exfiltration"] = df["data-exfiltration"].map({True: "Yes", False: "No"})
df["operational_impact"] = df["impact.category.Operational Impact"].map({True: "Yes", False: "No"})
df["affected_pii"] = df["affected-data.category.PII"].map({True: "Yes", False: "No"})

# Working table: 2020 to 2024 rows and the focus fields only
focus_cols = ["incident_year", "attacker_primary", "exfiltration", "operational_impact", "affected_pii"]
df_focus = df.loc[df["incident_year"].between(2020, 2024), focus_cols].reset_index(drop=True)

clean_summary = pd.DataFrame({
    "filled": df_focus.notna().sum(),
    "missing": df_focus.isna().sum(),
})
clean_summary

In [ ]:
df_focus.head()

In [ ]:
# Data-quality check for the focus columns, 2020 to 2024 only, from the full file
focus_raw = {
    "incident-date": "incident_year",
    "attacker": "attacker_primary",
    "data-exfiltration": "exfiltration",
    "impact.category.Operational Impact": "operational_impact",
    "affected-data.category.PII": "affected_pii",
}
raw_year = pd.to_datetime(raw_data["incident-date"], errors="coerce").dt.year
window = raw_data[raw_year.between(2020, 2024)]
focus_check = pd.DataFrame({
    "cleaned column": list(focus_raw.values()),
    "rows in 2020 to 2024": len(window),
    "blank values": [int(window[c].isna().sum()) for c in focus_raw],
    "no attacker or False": [
        int((window[c] == "[]").sum()) if c == "attacker"
        else int((window[c] == False).sum()) if c.startswith(("data-", "impact", "affected"))
        else 0
        for c in focus_raw
    ],
}, index=pd.Index(list(focus_raw.keys()), name="source column"))
focus_check

The cleaning keeps only the incident date, attacker, exfiltration, operational impact, and personal data affected, and drops the other 138 columns. The year comes from the incident date, and the three True/False fields become Yes/No text so a missing value is never read as False. The attacker list is parsed and the first name becomes the primary attacker, with empty lists labeled "Unattributed" rather than dropped. The working table then keeps the 3,359 incidents from 2020 to 2024. Every focus column is filled for all 3,359 rows. The file has no blank True/False values, so a False could mean "not reported" rather than "did not happen." Of the 3,359 incidents, 1,619 (48%) have no attacker, 1,773 are False for exfiltration, 1,658 for operational impact, and 2,119 for personal data affected.


## 3. Exploratory analysis

### 3a. Incidents per year

In [ ]:
per_year = df_focus["incident_year"].value_counts().sort_index().rename("incidents").to_frame()
per_year

In [ ]:
year_chart = (
    alt.Chart(per_year.reset_index())
    .mark_bar(color="#4e79a7")
    .encode(
        x=alt.X("incident_year:O", title="Year"),
        y=alt.Y("incidents:Q", title="Incidents"),
        tooltip=["incident_year:O", "incidents:Q"],
    )
    .properties(width=480, height=280, title="Incidents per year, 2020 to 2024")
)
year_chart

**Chart justification:** bar length on a shared baseline shows the count for each year, so the year-to-year change is easy to compare. One hue is used, since the year is the only grouping.

### 3b. Top attributed ransomware groups

In [ ]:
df_focus["attacker_primary"].value_counts().head(12).rename("incidents").to_frame()

In [ ]:
groups = df_focus["attacker_primary"].value_counts().head(12).rename("incidents").reset_index()
groups.columns = ["attacker_primary", "incidents"]
group_chart = (
    alt.Chart(groups)
    .mark_bar()
    .encode(
        y=alt.Y("attacker_primary:N", sort="-x", title="Attacker group"),
        x=alt.X("incidents:Q", title="Incidents"),
        color=alt.condition(
            alt.datum.attacker_primary == "Unattributed",
            alt.value("#999999"),
            alt.value("#4e79a7"),
        ),
        tooltip=["attacker_primary:N", "incidents:Q"],
    )
    .properties(width=480, height=300, title="Top attacker groups, including Unattributed")
)
group_chart


**Chart justification:** horizontal bars sorted by count keep the group names readable. Gray marks "Unattributed", the only no-data category, so it is visibly different from the named groups in blue.

### 3c. Data exfiltration

In [ ]:
rd_exfil = df_focus["exfiltration"].value_counts().rename("incidents").to_frame()
rd_exfil["share"] = (rd_exfil["incidents"] / len(df_focus)).round(3)
rd_exfil

In [ ]:
rd_exfil_chart = (
    alt.Chart(rd_exfil.reset_index())
    .mark_bar()
    .encode(
        x=alt.X("exfiltration:N", title="Data exfiltrated", sort=["Yes", "No"]),
        y=alt.Y("share:Q", title="Share of incidents", axis=alt.Axis(format="%")),
        color=alt.Color("exfiltration:N", title="Data exfiltrated", scale=alt.Scale(range=["#e15759", "#4e79a7"])),
        tooltip=["exfiltration:N", "incidents:Q", alt.Tooltip("share:Q", format=".1%")],
    )
    .properties(width=320, height=260, title="Data exfiltration, 2020 to 2024")
)
rd_exfil_chart

**Chart justification:** color separates Yes (red) from No (blue), and bar height shows the share, matching the table above.

### 3d. Operational impact

In [ ]:
rd_impact = df_focus["operational_impact"].value_counts().rename("incidents").to_frame()
rd_impact["share"] = (rd_impact["incidents"] / len(df_focus)).round(3)
rd_impact

In [ ]:
rd_impact_chart = (
    alt.Chart(rd_impact.reset_index())
    .mark_bar()
    .encode(
        x=alt.X("operational_impact:N", title="Operational impact", sort=["Yes", "No"]),
        y=alt.Y("share:Q", title="Share of incidents", axis=alt.Axis(format="%")),
        color=alt.Color("operational_impact:N", title="Operational impact", scale=alt.Scale(range=["#e15759", "#4e79a7"])),
        tooltip=["operational_impact:N", "incidents:Q", alt.Tooltip("share:Q", format=".1%")],
    )
    .properties(width=320, height=260, title="Operational impact, 2020 to 2024")
)
rd_impact_chart

**Chart justification:** color separates Yes (red) from No (blue), and bar height shows the share, matching the table in 3d.

### 3e. Personal data affected

In [ ]:
rd_pii = df_focus["affected_pii"].value_counts().rename("incidents").to_frame()
rd_pii["share"] = (rd_pii["incidents"] / len(df_focus)).round(3)
rd_pii

In [ ]:
rd_pii_chart = (
    alt.Chart(rd_pii.reset_index())
    .mark_bar()
    .encode(
        x=alt.X("affected_pii:N", title="Personal data affected", sort=["Yes", "No"]),
        y=alt.Y("share:Q", title="Share of incidents", axis=alt.Axis(format="%")),
        color=alt.Color("affected_pii:N", title="Personal data affected", scale=alt.Scale(range=["#e15759", "#4e79a7"])),
        tooltip=["affected_pii:N", "incidents:Q", alt.Tooltip("share:Q", format=".1%")],
    )
    .properties(width=320, height=260, title="Personal data affected, 2020 to 2024")
)
rd_pii_chart

**Chart justification:** color separates Yes (red) from No (blue), and bar height shows the share.

## 4. Visualization proposals

Each idea gives the question, the variables and computation, the encodings, the chart type, and the design. Idea 1 is built above. Ideas 2 to 5 are proposals for the group to discuss and are not built yet. All shares use the 2020 to 2024 rows (3,359 incidents), including the 48% with no attacker.

**Shared group rule (ideas 2 to 4).** Group `attacker_primary`. Keep the 10 groups with the most incidents, plus "Unattributed" as its own row. For each group compute incidents = row count, yes = count of "Yes" in the field, and share = yes ÷ incidents. Show incidents and share in the tooltip.

**1. Share of incidents by pattern (built, bar chart).**
- *Question:* how common are exfiltration, operational impact, personal data affected, and an unknown attacker?
- *Variables and computation:* `exfiltration`, `operational_impact`, `affected_pii` (Yes share each) and `attacker_primary` (Unattributed share). Each is a count ÷ 3,359.
- *Encodings:* y = pattern (`:N`), x = share (`:Q`), color = blue for the three Yes shares and gray for the unattributed share.
- *Chart type:* bar chart, one bar per pattern.
- *Design:* x = percent from 0 to 100%, ticks every 20%. Title "Share of 2020 to 2024 incidents by pattern". Bar labels show each percent. Size 480 × 200.

**2. Exfiltration share by top attacker groups (dot plot).**
- *Question:* which groups more often steal data?
- *Variables and computation:* the shared group rule on `exfiltration`.
- *Encodings:* y = group (`:N`), sorted by share descending; x = share (`:Q`); dot size = incidents (`:Q`); color = blue, gray for "Unattributed".
- *Chart type:* dot plot.
- *Design:* x = percent from 0 to 100%, ticks every 10%. Dot size legend at the smallest, median, and largest incident count. Each dot carries its percent label. Title "Exfiltration share by top attacker groups, 2020 to 2024". Size 480 wide, height about 25 px per group.

**3. Operational impact share by top attacker groups (dot plot).**
- *Question:* which groups more often disrupt operations?
- *Variables and computation:* the shared group rule on `operational_impact`.
- *Encodings:* same as idea 2.
- *Chart type:* dot plot.
- *Design:* same scales, ticks, colors, and size as idea 2, so the two charts compare by eye. Title "Operational impact share by top attacker groups, 2020 to 2024".

**4. Personal data affected share by top attacker groups (dot plot).**
- *Question:* which groups more often expose personal data?
- *Variables and computation:* the shared group rule on `affected_pii`.
- *Encodings:* same as idea 2.
- *Chart type:* dot plot.
- *Design:* same as idea 2. Title "Personal data affected share by top attacker groups, 2020 to 2024".

**5. Exfiltration against operational impact (heatmap of counts).**
- *Question:* do incidents that disrupt operations also steal data?
- *Variables and computation:* cross-tabulate `exfiltration` (Yes/No) by `operational_impact` (Yes/No). Counts only, no shares. Four cells sum to 3,359.
- *Encodings:* x = exfiltration (`:N`), y = operational impact (`:N`), color = count (`:Q`), cell text = count.
- *Chart type:* heatmap (rectangles colored by count).
- *Design:* single-hue blue scale from 0 to the largest cell, legend ticks at 0, half the maximum, and the maximum. x and y are category labels, so no numeric ticks. Title "Exfiltration against operational impact, 2020 to 2024". Size 320 × 260.